In [1]:
import json
import pandas as pd
import time
import requests
from bs4 import BeautifulSoup
import re

## **PROSES ARTIKEL MONGAAY INDONESIA**

In [2]:
file_json = "D:/TUGAS KAMPUS SEMESTER 6/DICODING/Capstone/artikel_capstone.json"

with open(file_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total data:", len(data))

for item in data[:3]:
    print(item)

df = pd.DataFrame(data)

print(len(df))
df.head(2)

Total data: 84
{'keyword': 'bahaya limbah sampah', 'title': 'Para Pihak Ingatkan Risiko Proyek Energi Sampah Pemerintah', 'url': 'https://mongabay.co.id/2026/04/29/para-pihak-ingatkan-risiko-proyek-energi-sampah-pemerintah/', 'image': 'https://indomgb.s3.amazonaws.com/wp-content/uploads/2026/04/28140517/20230524_135555.jpg', 'date': '2026-04-29', 'source': 'Mongabay Indonesia'}
{'keyword': 'kelola sampah', 'title': 'Aksi Para Perempuan dari Kelola Ratusan Bank Sampah sampai Kembangkan Energi Surya', 'url': 'https://mongabay.co.id/2026/04/26/aksi-para-perempuan-dari-kelola-ratusan-bank-sampah-sampai-kembangkan-energi-surya/', 'image': 'https://indomgb.s3.amazonaws.com/wp-content/uploads/2026/04/25045017/Pengelolaan-Sampah-Organik-1.jpg', 'date': '2026-04-26', 'source': 'Mongabay Indonesia'}
{'keyword': 'kelola sampah', 'title': 'Aksi Nyata Kelompok Perempuan di Tengah Karut Marut Kelola Sampah Jakarta', 'url': 'https://mongabay.co.id/2026/04/20/aksi-nyata-kelompok-perempuan-di-tengah-ka

,keyword,title,url,image,date,source
0,bahaya limbah sampah,Para Pihak Ingatkan Risiko Proyek Energi Sampa...,https://mongabay.co.id/2026/04/29/para-pihak-i...,https://indomgb.s3.amazonaws.com/wp-content/up...,2026-04-29,Mongabay Indonesia
1,kelola sampah,Aksi Para Perempuan dari Kelola Ratusan Bank S...,https://mongabay.co.id/2026/04/26/aksi-para-pe...,https://indomgb.s3.amazonaws.com/wp-content/up...,2026-04-26,Mongabay Indonesia


### **EXPLORATORY DATA ANALYSIS**

In [3]:
jumlah_duplikat = df.duplicated(subset="title").sum()
print("Jumlah title duplikat:", jumlah_duplikat)

duplikat_df = df[df.duplicated(subset="title", keep=False)]
duplikat_df = duplikat_df.sort_values("title")
duplikat_df

Jumlah title duplikat: 0


,keyword,title,url,image,date,source


In [4]:
missing_values = df.isnull().sum()
print("Jumlah missing values per kolom:")
print(missing_values)

Jumlah missing values per kolom:
keyword     0
title       0
url         0
image       0
date       16
source      0
dtype: int64


In [5]:
df_null_date = df[df["date"].isnull()]
print("Jumlah data dengan date kosong:", len(df_null_date))
df_null_date

Jumlah data dengan date kosong: 16


,keyword,title,url,image,date,source
68,green-info,Jejak Iklim di Balik Sampah Kita,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
69,green-info,Hujan Hari Ini Bawa Berkah… dan Plastik. Loh K...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
70,green-info,Alternatif Sehat dan Ramah Lingkungan Penggant...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
71,green-info,"Petugas Sampah, Pahlawan Lingkungan yang Serin...",https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
72,green-info,Optimalisasi Sistem Pengolahan Sampah Sebagai ...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
73,green-info,Mengintegrasikan Circular Economy dalam Upaya ...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
74,green-info,Peluang Menggerakkan Ekonomi Sirkular Melalui ...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
75,green-info,Kelola Sampah Demi Menyelamatkan Lingkungan,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
76,green-info,Keberhasilan Pengelolaan Sampah: Pentingnya Ko...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation
77,green-info,Sampah Sungai Citarum Langkah Awal GFDP Pulihk...,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation


### **PELABLEAN**

In [6]:
kategori_list = ["B3", "Glass", "Metal", "Organic", "Paper", "Plastic"]

kategori_keywords = {
    "B3": [
        "b3", "baterai", "oli", "kimia", "berbahaya",
        "racun", "limbah medis", "medis", "elektronik",
        "e-waste", "aki", "limbah industri", "nuklir"
    ],

    "Glass": [
        "kaca", "gelas", "botol kaca"
    ],

    "Metal": [
        "logam", "besi", "aluminium", "kaleng",
        "metal", "baja"
    ],

    "Organic": [
        "organik", "kompos", "daun", "makanan",
        "sisa makanan", "limbah makanan",
        "sampah makanan", "pangan", "pertanian", "gas metana"
    ],

    "Paper": [
        "kertas", "koran", "karton",
        "majalah", "buku"
    ],

    "Plastic": [
        "plastik", "botol plastik", "kresek",
        "mikroplastik", "kemasan",
        "kantong plastik", "sedotan"
    ]
}

def label_sampah(title):
    title = str(title).lower()
    
    for kategori, keywords in kategori_keywords.items():
        for kw in keywords:
            if kw in title:
                return kategori
    
    return "General"

df["Jenis Sampah"] = df["title"].apply(label_sampah)

df[["title", "Jenis Sampah"]].head()

,title,Jenis Sampah
0,Para Pihak Ingatkan Risiko Proyek Energi Sampa...,General
1,Aksi Para Perempuan dari Kelola Ratusan Bank S...,General
2,Aksi Nyata Kelompok Perempuan di Tengah Karut ...,General
3,Warga Klumprik Surabaya Tidak Lagi Resah Urusa...,General
4,Cuan Maggot dari Sampah Organik di Jakarta,Organic


In [7]:
frekuensi = df["Jenis Sampah"].value_counts()

print("Frekuensi per kategori:")
print(frekuensi)

Frekuensi per kategori:
Jenis Sampah
General    58
Plastic    17
Organic     6
Paper       2
B3          1
Name: count, dtype: int64


In [8]:
headers = {"User-Agent": "Mozilla/5.0"}

all_keywords = [k.lower() for kws in kategori_keywords.values() for k in kws]

df_general = df[df["Jenis Sampah"] == "General"]

results_cell7 = []

for idx, row in df_general.iterrows():
    url = row["url"]
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        
        # ambil lebih banyak elemen (lebih stabil dari hanya <p>)
        elements = soup.find_all(["p", "div", "article"])
        text = " ".join(e.get_text(" ") for e in elements).lower()
        
        # normalisasi spasi
        text = re.sub(r"\s+", " ", text)
        
        found = [kw for kw in all_keywords if kw in text]
        
        if found:
            results_cell7.append({
                "idx": idx,
                "title": row["title"],
                "text": text,
                "keywords": found
            })
            
            print(f"{row['title']} | {', '.join(found)}")
    
    except Exception as e:
        print(f"{row['title']} | ERROR")

Aksi Nyata Kelompok Perempuan di Tengah Karut Marut Kelola Sampah Jakarta | b3
Warga Klumprik Surabaya Tidak Lagi Resah Urusan Sampah | b3
Jejak Iklim di Balik Sampah Kita | oli, aki, kaca, organik, pangan, plastik, kemasan
Alternatif Sehat dan Ramah Lingkungan Pengganti Bakar Sampah | b3, berbahaya, racun, aki, kaca, logam, kaleng, organik, kompos, daun, makanan, sisa makanan, plastik, botol plastik
Petugas Sampah, Pahlawan Lingkungan yang Sering Terlupakan | b3, berbahaya, medis, elektronik, aki, organik, kompos, plastik
Optimalisasi Sistem Pengolahan Sampah Sebagai Strategi Meminimalisir Penumpukan Sampah dan Pariwisata Berkelanjutan di Kawasan Ekonomi Kreatif Mandalika, Lombok Tengah, NTB | oli, aki, organik, kompos, makanan, limbah makanan, pangan, pertanian, plastik
Mengintegrasikan Circular Economy dalam Upaya Green Contribution: Langkah Menuju Ekonomi Berkelanjutan Indonesia | berbahaya, elektronik, e-waste, logam, kaleng, organik, pertanian, plastik, botol plastik, kantong pla

In [9]:
df_filtered = df[df["title"] == "Jejak Iklim di Balik Sampah Kita"]
df_filtered

,keyword,title,url,image,date,source,Jenis Sampah
68,green-info,Jejak Iklim di Balik Sampah Kita,https://greeneration.org/publication/green-inf...,https://gfweb-dev.s3.ap-southeast-2.amazonaws....,None,Greeneration Foundation,General


In [10]:
def count_keywords(text, keywords):
    return sum(text.count(kw) for kw in keywords)


results_cell8 = []

for item in results_cell7:
    
    score = {}
    
    print("\n" + "="*80)
    print(item["title"])
    
    # hitung per kategori
    for kategori, kws in kategori_keywords.items():
        freq = count_keywords(item["text"], kws)
        score[kategori] = freq
        
        if freq > 0:
            print(f"{kategori} keywords = {freq}")
    
    # ambil kategori terbesar
    best_label = max(score, key=score.get)
    
    if score[best_label] == 0:
        best_label = "General"
    
    # simpan hasil (TIDAK update df)
    results_cell8.append({
        "idx": item["idx"],
        "title": item["title"],
        "label": best_label,
        "score": score
    })
    
    print("LABEL BARU (TEMP):", best_label)


Aksi Nyata Kelompok Perempuan di Tengah Karut Marut Kelola Sampah Jakarta
B3 keywords = 4
LABEL BARU (TEMP): B3

Warga Klumprik Surabaya Tidak Lagi Resah Urusan Sampah
B3 keywords = 4
LABEL BARU (TEMP): B3

Jejak Iklim di Balik Sampah Kita
B3 keywords = 54
Glass keywords = 9
Organic keywords = 35
Plastic keywords = 63
LABEL BARU (TEMP): Plastic

Alternatif Sehat dan Ramah Lingkungan Pengganti Bakar Sampah
B3 keywords = 63
Glass keywords = 9
Metal keywords = 18
Organic keywords = 114
Plastic keywords = 18
LABEL BARU (TEMP): Organic

Petugas Sampah, Pahlawan Lingkungan yang Sering Terlupakan
B3 keywords = 61
Organic keywords = 50
Plastic keywords = 17
LABEL BARU (TEMP): B3

Optimalisasi Sistem Pengolahan Sampah Sebagai Strategi Meminimalisir Penumpukan Sampah dan Pariwisata Berkelanjutan di Kawasan Ekonomi Kreatif Mandalika, Lombok Tengah, NTB
B3 keywords = 72
Organic keywords = 153
Plastic keywords = 8
LABEL BARU (TEMP): Organic

Mengintegrasikan Circular Economy dalam Upaya Green Cont

In [11]:
MIN_KEYWORD = 20

updated_count = 0

for item in results_cell8:
    
    idx = item["idx"]
    score = item["score"]
    
    best_category = max(score, key=score.get)
    best_score = score[best_category]
    
    if best_score >= MIN_KEYWORD:
        new_label = best_category
    else:
        new_label = "General"
    
    old_label = df.loc[idx, "Jenis Sampah"]
    
    if old_label != new_label:
        df.loc[idx, "Jenis Sampah"] = new_label
        updated_count += 1
        
        print("="*80)
        print(df.loc[idx, "title"])
        print(f"OLD LABEL : {old_label}")
        print(f"NEW LABEL : {new_label}")
        print(f"BEST SCORE: {best_score}")

print("\nTOTAL DATA UPDATED:", updated_count)

Jejak Iklim di Balik Sampah Kita
OLD LABEL : General
NEW LABEL : Plastic
BEST SCORE: 63
Alternatif Sehat dan Ramah Lingkungan Pengganti Bakar Sampah
OLD LABEL : General
NEW LABEL : Organic
BEST SCORE: 114
Petugas Sampah, Pahlawan Lingkungan yang Sering Terlupakan
OLD LABEL : General
NEW LABEL : B3
BEST SCORE: 61
Optimalisasi Sistem Pengolahan Sampah Sebagai Strategi Meminimalisir Penumpukan Sampah dan Pariwisata Berkelanjutan di Kawasan Ekonomi Kreatif Mandalika, Lombok Tengah, NTB
OLD LABEL : General
NEW LABEL : Organic
BEST SCORE: 153
Mengintegrasikan Circular Economy dalam Upaya Green Contribution: Langkah Menuju Ekonomi Berkelanjutan Indonesia
OLD LABEL : General
NEW LABEL : Plastic
BEST SCORE: 71
Peluang Menggerakkan Ekonomi Sirkular Melalui Pengelolaan Sampah-Polusi menjadi Energi
OLD LABEL : General
NEW LABEL : Organic
BEST SCORE: 225
Kelola Sampah Demi Menyelamatkan Lingkungan
OLD LABEL : General
NEW LABEL : Organic
BEST SCORE: 99
Sampah Sungai Citarum Langkah Awal GFDP Pulihka

In [12]:
df["Jenis Sampah"].value_counts()

Jenis Sampah
General    48
Plastic    20
Organic    11
B3          3
Paper       2
Name: count, dtype: int64

## **PROSES ARTIKEL EKUATORIAL**

In [13]:
headers = {"User-Agent": "Mozilla/5.0"}

keywords = {
    "paper": [
        "kertas", "sampah kertas", "limbah kertas", "kertas bekas",
        "kertas daur ulang", "daur ulang kertas", "kertas koran",
        "koran bekas", "majalah bekas", "buku bekas",
        "kertas kantor", "dokumen bekas", "arsip kertas",
        "kertas karton", "karton bekas", "cardboard",
        "bungkus kertas", "kemasan kertas", "paper packaging",
        "tissue", "tisu bekas", "tisu sekali pakai",
        "kertas tercampur sampah", "pemilahan kertas",
        "pengelolaan sampah kertas", "recycling kertas",
        "industri kertas", "limbah industri kertas",
        "sampah anorganik kertas"
    ],

    "glass": [
        "glass", "kaca", "botol kaca", "pecahan kaca", "cermin",
        "gelas", "botol minuman", "wadah kaca", "kaca bekas",
        "pecahan botol", "glass waste", "tempered glass",
        "kaca bangunan", "kaca jendela", "kaca daur ulang",
        "glass recycling"
    ],

    "metal": [
        "metal", "logam", "besi", "aluminium", "kaleng", "steel",
        "baja", "tembaga", "kuningan", "stainless", "timah", "nikel",
        "logam berat", "scrap", "rongsokan", "besi tua",
        "kaleng minuman", "kaleng makanan", "logam bekas",
        "metal waste", "metal scrap", "aluminium foil",
        "besi scrap", "logam daur ulang"
    ],

    "b3": [
        "b3", "berbahaya", "toxic", "racun", "kimia",
        "limbah berbahaya", "limbah medis", "elektronik",
        "e-waste", "aki", "baterai", "radioaktif"
    ]
}

base_url = "https://www.ekuatorial.com/kategori/tipe/artikel/page/{}/"

results = []

def is_noise_article(text):
    blacklist = [
        "mahkamah", "hakim", "konstitusi",
        "sidang", "kriminalisasi",
        "putusan pengadilan", "politik hukum"
    ]
    return any(x in text for x in blacklist)

# =========================
# NORMALIZER (FIX "semakin" BUG)
# =========================
def normalize(text):
    return re.sub(r'[^a-z0-9\s]', ' ', text.lower())

for page in range(1, 40):
    url = base_url.format(page)
    print(f"\nHalaman {page}")

    try:
        res = requests.get(url, headers=headers, timeout=10)

        if res.status_code != 200:
            print(f"Skip halaman {page} (status {res.status_code})")
            continue

        soup = BeautifulSoup(res.text, "html.parser")
        articles = soup.find_all("article", class_="category-card")

        print(f"Total artikel ditemukan: {len(articles)}")

        for art in articles:

            h2_tag = art.find("h2", class_="category-card-title")
            if not h2_tag:
                continue

            a_tag = h2_tag.find("a")
            if not a_tag:
                continue

            title = a_tag.get_text(strip=True)
            article_url = a_tag.get("href")

            # =========================
            # CLEAN TITLE
            # =========================
            title_l = normalize(title)

            # =========================
            # FILTER NOISE
            # =========================
            if is_noise_article(title_l):
                continue

            # =========================
            # KEYWORD MATCH (SAFE WORD MATCH)
            # =========================
            matched_categories = []
            matched_keywords = []

            for cat, kws in keywords.items():
                for kw in kws:

                    kw_norm = normalize(kw)

                    # WORD-BOUNDARY SAFE MATCH
                    if re.search(rf"\b{re.escape(kw_norm)}\b", title_l):
                        matched_categories.append(cat)
                        matched_keywords.append(kw)
                        break

            if not matched_categories:
                continue

            # =========================
            # EXTRACT DATE
            # =========================
            time_tag = art.find("time")
            date_val = time_tag.get("datetime") if time_tag else None

            # =========================
            # EXTRACT IMAGE
            # =========================
            thumb_div = art.find("div", class_="category-card-thumb")
            img_tag = thumb_div.find("img") if thumb_div else None
            img_url = img_tag.get("src") or img_tag.get("data-src") if img_tag else None

            # =========================
            # SAVE
            # =========================
            results.append({
                "title": title,
                "url": article_url,
                "image": img_url,
                "date": date_val,
                "categories": list(set(matched_categories)),
                "keywords": list(set(matched_keywords))
            })

    except requests.exceptions.Timeout:
        print(f"Timeout halaman {page}, SKIP")
        continue

    except requests.exceptions.RequestException as e:
        print(f"Error halaman {page}: {e}")
        continue

    time.sleep(1)

print("\nSCRAPING SELESAI")
print("Total hasil:", len(results))


Halaman 1
Total artikel ditemukan: 9

Halaman 2
Total artikel ditemukan: 9

Halaman 3
Total artikel ditemukan: 9

Halaman 4
Total artikel ditemukan: 9

Halaman 5
Total artikel ditemukan: 9

Halaman 6
Total artikel ditemukan: 9

Halaman 7
Total artikel ditemukan: 9

Halaman 8
Total artikel ditemukan: 9

Halaman 9
Total artikel ditemukan: 9

Halaman 10
Total artikel ditemukan: 9

Halaman 11
Total artikel ditemukan: 9

Halaman 12
Total artikel ditemukan: 9

Halaman 13
Total artikel ditemukan: 9

Halaman 14
Total artikel ditemukan: 9

Halaman 15
Total artikel ditemukan: 9

Halaman 16
Total artikel ditemukan: 9

Halaman 17
Total artikel ditemukan: 9

Halaman 18
Total artikel ditemukan: 9

Halaman 19
Total artikel ditemukan: 9

Halaman 20
Total artikel ditemukan: 9

Halaman 21
Total artikel ditemukan: 9

Halaman 22
Total artikel ditemukan: 9

Halaman 23
Total artikel ditemukan: 9

Halaman 24
Total artikel ditemukan: 9

Halaman 25
Total artikel ditemukan: 9

Halaman 26
Total artikel ditemuka

In [14]:
print("HASIL ARTIKEL YANG MATCH")

if results:
    for i, r in enumerate(results, 1):
        print(f"{i}. {r['title']}")
        print(f"Kategori : {', '.join(r['categories'])}")
        print(f"Keyword  : {', '.join(r['keywords'])}\n")
else:
    print("Tidak ada artikel yang sesuai keyword.")

HASIL ARTIKEL YANG MATCH
1. Polutan Udara Berbahaya Mengintai Kesehatan Warga Lima Kota Besar di Indonesia
Kategori : b3
Keyword  : berbahaya

2. Temuan Lima Logam Berat Hantui Dasar Laut Teluk Jakarta
Kategori : metal
Keyword  : logam

3. Greenwashing: Paradoks Pendanaan ‘Hijau’ di Balik Industri Nikel Pulau Obi
Kategori : metal
Keyword  : nikel

4. Monster Baja Mengepung, Masyarakat Adat Imekko Papua Aktifkan Alarm Siaga
Kategori : metal
Keyword  : baja

5. Jejak Racun dan Ketidakadilan di Pulau Kabaena
Kategori : b3
Keyword  : racun

6. Siapa Paling Untung dari Megaproyek Baterai EV Indonesia?
Kategori : b3
Keyword  : baterai

7. Sanksi yang Belum Nyata Pasca Longsor Limbah Nikel QMB di Morowali
Kategori : metal
Keyword  : nikel

8. Kajian IPB Terkait Kematian Massal Ikan Dewa di Kuningan
Kategori : metal
Keyword  : kuningan

9. Menjinakkan ledakan sampah elektronik Indonesia
Kategori : b3
Keyword  : elektronik

10. Benarkah nikel kita hijau? Menggugat keadilan di balik baterai kend

In [15]:
import re


positive_waste_keywords = [
    "sampah", "limbah", "daur ulang", "recycle",
    "waste", "tpa", "tpst", "maggot",
    "plastik", "kertas", "kaca", "logam",
    "b3", "beracun", "e-waste", "elektronik",
    "kompos"
]


noise_keywords = [
    "investasi", "megaproyek", "ekonomi", "industri baterai",
    "psn", "politik", "hukum", "konstitusi",
    "hakim", "sidang", "kriminalisasi",
    "pln", "energi", "nze",
    "komoditas mahal", "air jadi komoditas",
    "krisis ekologis", "perubahan iklim",
    "jurnalisme", "narasi", "keadilan"
]

def is_valid_waste_article(title):

    t = title.lower()

    # 1. HARUS ada indikasi sampah/limbah
    has_waste_signal = any(k in t for k in positive_waste_keywords)

    # 2. BUANG kalau terlalu politik/ekonomi/energi
    has_noise = any(k in t for k in noise_keywords)

    # RULE FINAL
    return has_waste_signal and not has_noise


filtered_results = []

for r in results:
    if is_valid_waste_article(r["title"]):
        filtered_results.append(r)

print("SEBELUM FILTER:", len(results))
print("SESUDAH FILTER :", len(filtered_results))
filtered_results

SEBELUM FILTER: 13
SESUDAH FILTER : 3


[{'title': 'Temuan Lima Logam Berat Hantui Dasar Laut Teluk Jakarta',
  'url': 'https://www.ekuatorial.com/2026/05/temuan-lima-logam-berat-hantui-dasar-laut-teluk-jakarta/',
  'image': 'https://www.ekuatorial.com/wp-content/uploads/2026/05/Sailboat_Jakarta_bay-1024x768.avif',
  'date': '2026-05-12T11:50:12+07:00',
  'categories': ['metal'],
  'keywords': ['logam']},
 {'title': 'Sanksi yang Belum Nyata Pasca Longsor Limbah Nikel QMB di Morowali',
  'url': 'https://www.ekuatorial.com/2026/03/sanksi-yang-belum-nyata-pasca-longsor-limbah-nikel-qmb-di-morowali/',
  'image': 'https://www.ekuatorial.com/wp-content/uploads/2026/03/HAEL.avif',
  'date': '2026-03-04T14:51:28+07:00',
  'categories': ['metal'],
  'keywords': ['nikel']},
 {'title': 'Menjinakkan ledakan sampah elektronik Indonesia',
  'url': 'https://www.ekuatorial.com/2025/12/menjinakkan-ledakan-sampah-elektronik-indonesia/',
  'image': 'https://www.ekuatorial.com/wp-content/uploads/2025/12/Sampah-elektronik-i-Indonesia-diprediksi-

In [16]:
df_new = pd.DataFrame(filtered_results)
RELEVANT_KEYWORDS = [
    "sampah",
    "daur ulang",
    "plastik",
    "bank sampah",
    "pemilahan",
    "kompos",
    "guna ulang",
    "reduce",
    "reuse",
    "recycle",
    "pengelolaan",
    "limbah plastik",
    "sampah plastik",
    "tpa",
    "tpst",
    "upcycle",
    "eco",
]

def dapatkan_keyword(title):
    if not isinstance(title, str):
        return "Lainnya"

    title_lowercase = title.lower()

    for kw in RELEVANT_KEYWORDS:
        if kw in title_lowercase:
            return kw.capitalize() 

    return "Lainnya"  

if not df_new.empty:
    df_new["keyword"] = df_new["title"].apply(dapatkan_keyword)

    df_new["Jenis Sampah"] = df_new["categories"].apply(
        lambda x: x[0].capitalize() if isinstance(x, list) and len(x) > 0 else "General"
    )

    df_new["source"] = "Ekuatorial"

    df_new["date"] = pd.to_datetime(df_new["date"]).dt.strftime("%Y-%m-%d")
else:
    df_new = pd.DataFrame(
        columns=[
            "keyword",
            "title",
            "url",
            "image",
            "date",
            "source",
            "Jenis Sampah",
        ]
    )

df_new = df_new[
    ["keyword", "title", "url", "image", "date", "source", "Jenis Sampah"]
]

df_new

,keyword,title,url,image,date,source,Jenis Sampah
0,Lainnya,Temuan Lima Logam Berat Hantui Dasar Laut Telu...,https://www.ekuatorial.com/2026/05/temuan-lima...,https://www.ekuatorial.com/wp-content/uploads/...,2026-05-12,Ekuatorial,Metal
1,Lainnya,Sanksi yang Belum Nyata Pasca Longsor Limbah N...,https://www.ekuatorial.com/2026/03/sanksi-yang...,https://www.ekuatorial.com/wp-content/uploads/...,2026-03-04,Ekuatorial,Metal
2,Sampah,Menjinakkan ledakan sampah elektronik Indonesia,https://www.ekuatorial.com/2025/12/menjinakkan...,https://www.ekuatorial.com/wp-content/uploads/...,2025-12-26,Ekuatorial,B3


In [17]:
df_new["Jenis Sampah"].value_counts()

Jenis Sampah
Metal    2
B3       1
Name: count, dtype: int64

## **GABUNG KEDUA ARTIKEL**

In [18]:
df_final = pd.concat([df, df_new], ignore_index=True)

# (opsional) pastikan kolom tetap rapi & konsisten
df_final = df_final[
    ["keyword", "title", "url", "image", "date", "source", "Jenis Sampah"]
]

df_final.head(2)

,keyword,title,url,image,date,source,Jenis Sampah
0,bahaya limbah sampah,Para Pihak Ingatkan Risiko Proyek Energi Sampa...,https://mongabay.co.id/2026/04/29/para-pihak-i...,https://indomgb.s3.amazonaws.com/wp-content/up...,2026-04-29,Mongabay Indonesia,General
1,kelola sampah,Aksi Para Perempuan dari Kelola Ratusan Bank S...,https://mongabay.co.id/2026/04/26/aksi-para-pe...,https://indomgb.s3.amazonaws.com/wp-content/up...,2026-04-26,Mongabay Indonesia,General


In [19]:
df_final.isnull().sum()

keyword          0
title            0
url              0
image            0
date            16
source           0
Jenis Sampah     0
dtype: int64

In [20]:
df_final[df_final["date"].isnull()][["title", "date", "Jenis Sampah"]]

,title,date,Jenis Sampah
68,Jejak Iklim di Balik Sampah Kita,None,Plastic
69,Hujan Hari Ini Bawa Berkah… dan Plastik. Loh K...,None,Plastic
70,Alternatif Sehat dan Ramah Lingkungan Penggant...,None,Organic
71,"Petugas Sampah, Pahlawan Lingkungan yang Serin...",None,B3
72,Optimalisasi Sistem Pengolahan Sampah Sebagai ...,None,Organic
73,Mengintegrasikan Circular Economy dalam Upaya ...,None,Plastic
74,Peluang Menggerakkan Ekonomi Sirkular Melalui ...,None,Organic
75,Kelola Sampah Demi Menyelamatkan Lingkungan,None,Organic
76,Keberhasilan Pengelolaan Sampah: Pentingnya Ko...,None,General
77,Sampah Sungai Citarum Langkah Awal GFDP Pulihk...,None,B3


In [21]:
df_final = df_final.dropna(subset=["date"]).reset_index(drop=True)
len(df_final)

71

In [22]:
df_final["Jenis Sampah"].value_counts()

Jenis Sampah
General    46
Plastic    13
Organic     6
B3          2
Paper       2
Metal       2
Name: count, dtype: int64

In [23]:
df_filtered = df_final[
    df_final["Jenis Sampah"].str.lower().isin(
        ["metal", "organic", "glass", "paper", "b3"]
    )
]

df_sorted = df_filtered.sort_values("Jenis Sampah")

df_sorted[["title", "Jenis Sampah"]].to_csv("df_sorted.csv", index=False)


In [24]:
df_final.duplicated().sum()

np.int64(0)

## **TAMBAHAN ARTIKEL PAPER**

In [ ]:
import pandas as pd
import re

def extract_img_url(text):
    if not isinstance(text, str):
        return None
    
    match = re.search(r'src="(https?://[^"]+)"', text)
    if match:
        return match.group(1)
    
    match = re.search(r'https?://[^\s"\']+\.(jpg|jpeg|png|webp)', text)
    if match:
        return match.group(0)
    
    return text


artikel_1 = {
    "keyword": "Sampah",
    "title": "Sampah Kertas dan Bahayanya Terhadap Lingkungan",
    "url": "https://universaleco.id/sampah-kertas-dan-bahayanya-terhadap-lingkungan/",
    "image": "https://universaleco.id/wp-content/uploads/2025/07/steptodown.com786743.jpg",
    "date": "03-07-2025",
    "source": "Universal Eco",
    "Jenis Sampah": "Paper"
}

artikel_2 = {
    "keyword": "Pengelolaan",
    "title": "Limbah Kertas Tidak Dikelola? Siap-Siap Bahaya Mengintai!",
    "url": "https://kahuripancitra.com/limbah-kertas-tidak-dikelola-siap-siap-bahaya-mengintai/",
    "image": "https://kahuripancitra.com/wp-content/uploads/2024/02/2148996918-scaled.jpg",
    "date": "27-02-2024",
    "source": "Kahuripan Citra",
    "Jenis Sampah": "Paper"
}

artikel_3 = {
    "keyword": "Sampah",
    "title": "Sampah Kaleng Bikin Resah? Jangan Dibuang, Ketahui Cara Mengatasinya",
    "url": "https://madanitek.com/blog/sampah-kaleng-dan-tips-penanggulangannya/",
    "image": "https://madanitek.com/wp-content/uploads/2025/11/Sampah-Kaleng--1024x682.jpg",
    "date": "01-11-2025",
    "source": "Madanitec",
    "Jenis Sampah": "Metal"
}

artikel_4 = {
    "keyword": "Sampah",
    "title": "Berharap Limbah Kaleng Tidak Berakhir di Sungai",
    "url": "https://eco.espos.id/berharap-limbah-kaleng-tidak-berakhir-di-sungai-2161635",
    "image": "https://imgcdn.espos.id/@espos/images/2025/11/20251111193717-ilustrasi-kaleng-bekas.png?quality=60",
    "date": "11-11-2025",
    "source": "Espos Eco",
    "Jenis Sampah": "Metal"
}

artikel_5 = {
    "keyword": "Pengelolaan",
    "title": "Pengolahan Limbah B3 yang Tepat: Cara, Contoh di Indonesia, dan Dampaknya",
    "url": "https://thisistbs.com/id/publikasi/kisah/pengelolaan-limbah-b3-yang-tepat",
    "image": "https://assets.thisistbs.com/pengolahan-limbah-b3.jpg",
    "date": "06-05-2026",
    "source": "TBS",
    "Jenis Sampah": "B3"
}

artikel_6 = {
    "keyword": "Pengelolaan",
    "title": "Syarat Pengolahan Limbah B3 secara Termal sesuai Peraturan",
    "url": "https://rcs.hukumonline.com/insights/pengolahan-limbah-b3-secara-termal",
    "image": "https://es-rcs.s3.ap-southeast-3.amazonaws.com/cms/pexels.jpg",
    "date": "10-06-2024",
    "source": "Regulatory Compliance System",
    "Jenis Sampah": "B3"
}

artikel_7 = {
    "keyword": "Sampah",
    "title": "Ancaman Lain dari Sampah Kaca",
    "url": "https://eco.espos.id/ancaman-lain-dari-sampah-kaca-2162137",
    "image": "https://imgcdn.espos.id/@espos/images/2025/11/20251112185159-ilustrasi-memilah-sampah-kaca.png?quality=60",
    "date": "13-11-2025",
    "source": "Espos Eco",
    "Jenis Sampah": "Glass"
}

artikel_8 = {
    "keyword": "Pengelolaan",
    "title": "Pemanfaatan Botol Kaca Bekas Jadi Vas Bunga, Solusi Kreatif Kurangi Sampah",
    "url": "https://eco.espos.id/pemanfaatan-botol-kaca-bekas-jadi-vas-bunga-solusi-kreatif-kurangi-sampah-2167335",
    "image": "https://imgcdn.espos.id/@espos/images/2025/11/20251127184416-botol-kaca.jpg?quality=60",
    "date": "27-11-2025",
    "source": "Espos Eco",
    "Jenis Sampah": "Glass"
}

artikel_9 = {
    "keyword": "Limbah",
    "title": "Pentingnya Proses Buang Limbah Kaca",
    "url": "https://rafikatransindo.co.id/buang-limbah-kaca/",
    "image": "https://rafikatransindo.co.id/wp-content/uploads/2024/08/Limbah-Kaca.jpg",
    "date": "01-01-2024",
    "source": "PT Rafika Trans Indonesia",
    "Jenis Sampah": "Glass"
}

artikel_10 = {
    "keyword": "Limbah",
    "title": "Limbah Kaca Bernilai Ekonomi, Sarana Edukasi Pengelolaan Sampah",
    "url": "https://www.bisnisjogja.id/limbah-kaca-bernilai-ekonomi-sarana-edukasi-pengelolaan-sampah/",
    "image": "https://www.bisnisjogja.id/wp-content/uploads/2025/09/24iLIM-1.webp",
    "date": "24-09-2025",
    "source": "Bisnis Jogja",
    "Jenis Sampah": "Glass"
}

artikel_11 = {
    "keyword": "Sampah",
    "title": "Kaleng yang Kita Buang, Racun yang Kita Tinggalkan",
    "url": "https://eco.espos.id/kaleng-yang-kita-buang-racun-yang-kita-tinggalkan-2164631",
    "image": "https://imgcdn.espos.id/@espos/images/2025/11/20251119173438-13-aina-foto-16.jpg?quality=60",
    "date": "19-11-2025",
    "source": "Espos Eco",
    "Jenis Sampah": "Metal"
}

df_manual = pd.DataFrame([
    artikel_1, artikel_2, artikel_3, artikel_4, artikel_5,
    artikel_6, artikel_7, artikel_8, artikel_9, artikel_10, artikel_11
])

df_manual["image"] = df_manual["image"].apply(extract_img_url)

df_manual = df_manual[
    ["keyword", "title", "url", "image", "date", "source", "Jenis Sampah"]
]

df_manual

,keyword,title,url,image,date,source,Jenis Sampah
0,Sampah,Sampah Kertas dan Bahayanya Terhadap Lingkungan,https://universaleco.id/sampah-kertas-dan-baha...,https://universaleco.id/wp-content/uploads/202...,03-07-2025,Universal Eco,Paper
1,Pengelolaan,Limbah Kertas Tidak Dikelola? Siap-Siap Bahaya...,https://kahuripancitra.com/limbah-kertas-tidak...,https://kahuripancitra.com/wp-content/uploads/...,27-02-2024,Kahuripan Citra,Paper
2,Sampah,"Sampah Kaleng Bikin Resah? Jangan Dibuang, Ket...",https://madanitek.com/blog/sampah-kaleng-dan-t...,https://madanitek.com/wp-content/uploads/2025/...,01-11-2025,Madanitec,Metal
3,Sampah,Berharap Limbah Kaleng Tidak Berakhir di Sungai,https://eco.espos.id/berharap-limbah-kaleng-ti...,https://imgcdn.espos.id/@espos/images/2025/11/...,11-11-2025,Espos Eco,Metal
4,Pengelolaan,"Pengolahan Limbah B3 yang Tepat: Cara, Contoh ...",https://thisistbs.com/id/publikasi/kisah/penge...,https://assets.thisistbs.com/pengolahan-limbah...,06-05-2026,TBS,B3
5,Pengelolaan,Syarat Pengolahan Limbah B3 secara Termal sesu...,https://rcs.hukumonline.com/insights/pengolaha...,https://es-rcs.s3.ap-southeast-3.amazonaws.com...,10-06-2024,Regulatory Compliance System,B3
6,Sampah,Ancaman Lain dari Sampah Kaca,https://eco.espos.id/ancaman-lain-dari-sampah-...,https://imgcdn.espos.id/@espos/images/2025/11/...,13-11-2025,Espos Eco,Glass
7,Pengelolaan,"Pemanfaatan Botol Kaca Bekas Jadi Vas Bunga, S...",https://eco.espos.id/pemanfaatan-botol-kaca-be...,https://imgcdn.espos.id/@espos/images/2025/11/...,27-11-2025,Espos Eco,Glass
8,Limbah,Pentingnya Proses Buang Limbah Kaca,https://rafikatransindo.co.id/buang-limbah-kaca/,https://rafikatransindo.co.id/wp-content/uploa...,01-01-2024,PT Rafika Trans Indonesia,Glass
9,Limbah,"Limbah Kaca Bernilai Ekonomi, Sarana Edukasi P...",https://www.bisnisjogja.id/limbah-kaca-bernila...,https://www.bisnisjogja.id/wp-content/uploads/...,24-09-2025,Bisnis Jogja,Glass


In [26]:
df_final = pd.concat([df_final, df_manual], ignore_index=True)

In [27]:
df_final.isnull().sum()

keyword         0
title           0
url             0
image           0
date            0
source          0
Jenis Sampah    0
dtype: int64

In [28]:
df_final["Jenis Sampah"].value_counts()

Jenis Sampah
General    46
Plastic    13
Organic     6
Metal       5
B3          4
Paper       4
Glass       4
Name: count, dtype: int64

In [30]:
import json

data = df_final.to_dict(orient="records")

with open("artikel_capstone_ecosort.json", "w", encoding="utf-8") as f:
    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=2
    )